# LAB | Intro to Machine Learning

**Load the data**

In this challenge, we will be working with Spaceship Titanic data. The data can be found here:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv

Metadata

https://github.com/data-bootcamp-v4/data/blob/main/spaceship_titanic.md

#### Spaceship Titanic Dataset
Metadata

- Id: Unique identifier for each property.
- PassengerId - A unique Id for each passenger. Each Id takes the form gggg_pp where gggg indicates a group the passenger is travelling with and pp is their number within the group. People in a group are often family members, but not always.
- HomePlanet - The planet the passenger departed from, typically their planet of permanent residence.
- CryoSleep - Indicates whether the passenger elected to be put into suspended animation for the duration of the voyage. Passengers in cryosleep are confined to their cabins.
- Cabin - The cabin number where the passenger is staying. Takes the form deck/num/side, where side can be either P for Port or S for Starboard.
- Destination - The planet the passenger will be debarking to.
- Age - The age of the passenger.
- VIP - Whether the passenger has paid for special VIP service during the voyage. RoomService, FoodCourt, ShoppingMall, Spa, VRDeck - Amount the passenger has billed at each of the Spaceship Titanic's many luxury amenities.
- Name - The first and last names of the passenger.
- Transported - Whether the passenger was transported to another dimension. This is the target, the column you are trying to predict.

In [19]:
#import libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report,
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)


import matplotlib.pyplot as plt
import seaborn as sns



sns.set_theme(style="whitegrid")

In [20]:
spaceship = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv")
spaceship.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


**Check the shape of your data**

In [21]:
# 2. Shape, hiányzó értékek és első sorok ellenőrzése
print(f"Shape: {spaceship.shape}")
print("Missing values:", spaceship.isna().sum().sum())
display(spaceship.head())

# 3. Class balance ellenőrzése (itt a célváltozó oszlop neve 'Transported')
class_counts = spaceship["Transported"].value_counts().sort_index()
print("Class counts:")
print(class_counts)

Shape: (8693, 14)
Missing values: 2324


,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


Class counts:
Transported
False    4315
True     4378
Name: count, dtype: int64


**Check for data types**

In [22]:
print(spaceship.dtypes)

PassengerId         str
HomePlanet          str
CryoSleep        object
Cabin               str
Destination         str
Age             float64
VIP              object
RoomService     float64
FoodCourt       float64
ShoppingMall    float64
Spa             float64
VRDeck          float64
Name                str
Transported        bool
dtype: object


**Check for missing values**

In [23]:
print("Missing values:", spaceship.isna().sum().sum())
print("="*60)
print(spaceship.isna().sum())

Missing values: 2324
PassengerId       0
HomePlanet      201
CryoSleep       217
Cabin           199
Destination     182
Age             179
VIP             203
RoomService     181
FoodCourt       183
ShoppingMall    208
Spa             183
VRDeck          188
Name            200
Transported       0
dtype: int64


There are multiple strategies to handle missing data

- Removing all rows or all columns containing missing data.
- Filling all missing values with a value (mean in continouos or mode in categorical for example).
- Filling all missing values with an algorithm.

For this exercise, because we have such low amount of null values, we will drop rows containing any missing value. 

In [24]:
spaceship_clean = spaceship.dropna().copy()

print(f"Original shape: {spaceship.shape}")
print(f"Clean shape: {spaceship_clean.shape}")
print("Remaining missing values:", spaceship_clean.isna().sum().sum())

Original shape: (8693, 14)
Clean shape: (6606, 14)
Remaining missing values: 0


**KNN**

K Nearest Neighbors is a distance based algorithm, and requeries all **input data to be numerical.**

Let's only select numerical columns as our features.

Based on the 'dtypes' check there are 6 numerical (float64) column in the dataframe:

- Age
- RoomService
- FoodCourt
- ShoppingMall
- Spa
- VRDeck

In [25]:
#only the numeric columns will be used for the model, so we will select only those
x_spaceship = spaceship_clean.select_dtypes(include="number")

And also lets define our target.

In [26]:
y_spaceship = spaceship_clean["Transported"]

In [27]:
print("Selected numerical features:", list(x_spaceship.columns))
print("Features shape:", x_spaceship.shape)
print("Target shape:", y_spaceship.shape)

Selected numerical features: ['Age', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']
Features shape: (6606, 6)
Target shape: (6606,)


**Train Test Split**

Now that we have split the data into **features** and **target** variables and imported the **train_test_split** function, split X and y into X_train, X_test, y_train, and y_test. 80% of the data should be in the training set and 20% in the test set.

In [28]:
#your code here

X_train_spaceship, X_test_spaceship, y_train_spaceship, y_test_spaceship = train_test_split(
    x_spaceship,
    y_spaceship,
    test_size=0.20,
    random_state=0,
    stratify=y_spaceship,
)

print("Training observations:", len(X_train_spaceship))
print("Test observations:", len(X_test_spaceship))
print("Training features:", list(X_train_spaceship.columns))

Training observations: 5284
Test observations: 1322
Training features: ['Age', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']


**Model Selection**

In this exercise we will be using **KNN** as our predictive model.

You need to choose between **Classificator** or **Regressor**. Take into consideration target variable to decide.

Initialize a KNN instance without setting any hyperparameter.

In [30]:
# 1. Create a KNN classifier instance
knn_classifier = KNeighborsClassifier()

# 2. Fit the classifier to the training data
knn_classifier.fit(X_train_spaceship, y_train_spaceship)

# 3. Predict the labels of the test data
y_pred_test_knn = knn_classifier.predict(X_test_spaceship)

# 4. Evaluate the model's performance using accuracy score
accuracy = accuracy_score(y_test_spaceship, y_pred_test_knn)
print(f"Test Accuracy: {accuracy:.4f}")

comparison_spaceship = pd.DataFrame({
    "actual_transported": y_test_spaceship.map({True: "Transported", False: "Not Transported"}),
    "predicted_transported": pd.Series(y_pred_test_knn, index=y_test_spaceship.index).map(
        {True: "Transported", False: "Not Transported"}
    )
})

display(comparison_spaceship.head(10))

Test Accuracy: 0.7814


,actual_transported,predicted_transported
2236,Not Transported,Not Transported
2523,Transported,Transported
2204,Transported,Transported
8069,Not Transported,Not Transported
404,Not Transported,Not Transported
7335,Not Transported,Not Transported
1941,Not Transported,Not Transported
8259,Transported,Not Transported
5737,Not Transported,Transported
1068,Not Transported,Transported


Fit the model to your data.

In [12]:
#your code here

Evaluate your model.

In [13]:
#your code here

**Congratulations, you have just developed your first Machine Learning model!**